# RiskLens: First look at the data

**Goal of this notebook:** get to know the Home Credit applications before building any model. I want to know how big the data is, how many people default, what's missing and which columns seem related to default.

**Data:** `application_train.csv` (one row per loan application, `TARGET = 1` means the person had trouble repaying)

**What I'll check**
1. Size of the data and the target balance
2. Missing values
3. Age and employment
4. Which features relate to default
5. Quick summary of what I learned

## 1. Loading the data

In [32]:
import pandas as pd

app = pd.read_csv("..\\data\\raw\\home-credit-default-risk\\application_train.csv")
app.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


The table has **[307,511] rows and [122] columns**. Each row is one loan application. There are a lot of columns, so I won't look at all of them. I'll focus on the ones that look useful for predicting default.

## 2. How many people default?

In [33]:
# how many people defaulted?
print(app["TARGET"].value_counts())
print(app["TARGET"].value_counts(normalize=True))

TARGET
0    282686
1     24825
Name: count, dtype: int64
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


### What the table looks like
- **307,511 rows** and **122 columns**, one row per loan application
- `TARGET` is the column I want to predict: **1 = had payment difficulties, 0 = repaid fine**
- That's a lot of columns, so I'll focus on the ones that look useful for predicting default

### The data is very imbalanced

| TARGET | Count | Share |
|---|---|---|
| 0 (no default) | 282,686 | 91.9% |
| 1 (default) | 24,825 | 8.1% |

Only about **8%** of the loans are defaults. A model that says "nobody defaults" every time would be right **91.9%** of the time and still catch zero risky customers. So accuracy is a bad metric here. I'll use **AUC** and **recall on the default class** instead, and probably handle the imbalance when training (class weights).

## 3. Missing values

In [23]:
# which columns have the most missing values
missing = app.isnull().mean().sort_values(ascending=False)
print(missing.head(15))

COMMONAREA_AVG              0.698723
COMMONAREA_MODE             0.698723
COMMONAREA_MEDI             0.698723
NONLIVINGAPARTMENTS_MEDI    0.694330
NONLIVINGAPARTMENTS_MODE    0.694330
NONLIVINGAPARTMENTS_AVG     0.694330
FONDKAPREMONT_MODE          0.683862
LIVINGAPARTMENTS_AVG        0.683550
LIVINGAPARTMENTS_MEDI       0.683550
LIVINGAPARTMENTS_MODE       0.683550
FLOORSMIN_MODE              0.678486
FLOORSMIN_AVG               0.678486
FLOORSMIN_MEDI              0.678486
YEARS_BUILD_AVG             0.664978
YEARS_BUILD_MODE            0.664978
dtype: float64


The columns with the most missing values are all about the **applicant's housing** (common area, living apartments, floors, year built...), with up to **~70%** missing (`COMMONAREA` is the worst at 69.9%).

Each of these shows up three times (`_AVG`, `_MODE`, `_MEDI`) with the *exact same* percentage missing. That makes me think they're three versions of the same information, so they're probably very correlated. Something to check later: maybe keep only one version per group.

How I'll handle it: tree models like LightGBM can deal with NaN directly, but logistic regression can't, so for the baseline I'll have to fill them (median) or drop the worst columns.

## 4. Age and employment

`DAYS_BIRTH` is stored as a negative number of days (days before the application), so I converted it to years.

In [24]:
# age is stored as negative days, so I convert it to years
app["AGE"] = -app["DAYS_BIRTH"] / 365
print(app["AGE"].describe())

count    307511.000000
mean         43.936973
std          11.956133
min          20.517808
25%          34.008219
50%          43.150685
75%          53.923288
max          69.120548
Name: AGE, dtype: float64


Applicants are between **20.5 and 69 years old**, average around **44**. Nothing strange here.

In [25]:
# default rate for each age group
app["AGE_GROUP"] = pd.cut(app["AGE"], bins=[20, 30, 40, 50, 60, 70])
print(app.groupby("AGE_GROUP")["TARGET"].mean())

AGE_GROUP
(20, 30]    0.114569
(30, 40]    0.095835
(40, 50]    0.076508
(50, 60]    0.061297
(60, 70]    0.049214
Name: TARGET, dtype: float64


### Default rate by age group

| Age group | Default rate |
|---|---|
| 20-30 | 11.5% |
| 30-40 | 9.6% |
| 40-50 | 7.7% |
| 50-60 | 6.1% |
| 60-70 | 4.9% |

The pattern is very clear: **the younger the applicant, the more likely they default**. The 20-30 group defaults more than twice as often as the 60-70 group.

*Note for later:* age is a sensitive variable in credit decisions, so I want to check how much the model relies on it and mention it in the fairness part of the project.

In [26]:
# DAYS_EMPLOYED looks weird, checking it
print(app["DAYS_EMPLOYED"].describe())
print((app["DAYS_EMPLOYED"] == 365243).sum())

count    307511.000000
mean      63815.045904
std      141275.766519
min      -17912.000000
25%       -2760.000000
50%       -1213.000000
75%        -289.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64
55374


### Weird value in DAYS_EMPLOYED

The max of `DAYS_EMPLOYED` is **365,243 days** (about 1,000 years), and it appears in **55,374 rows** (around 18% of the data). It obviously isn't a real number of days worked, so it looks like a placeholder for "unknown". It also messed up the mean (63,815 days, when the median is only -1,213).

I replaced it with `NaN` so it doesn't distort the model. I haven't checked yet who these 55k people are, so that's a to-do (looking at their `NAME_INCOME_TYPE` should tell me).

In [27]:
import numpy as np
app["DAYS_EMPLOYED"] = app["DAYS_EMPLOYED"].replace(365243, np.nan)

## 5. What relates to default?

Correlation with `TARGET`. I know this only captures linear relations and doesn't prove cause and effect, but it's a quick first filter.

In [28]:
# which columns are most related to the target
cols = ["TARGET", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "AGE", "AMT_INCOME_TOTAL", "AMT_CREDIT"]
print(app[cols].corr()["TARGET"])

TARGET              1.000000
EXT_SOURCE_1       -0.155317
EXT_SOURCE_2       -0.160472
EXT_SOURCE_3       -0.178919
AGE                -0.078239
AMT_INCOME_TOTAL   -0.003982
AMT_CREDIT         -0.030369
Name: TARGET, dtype: float64


In [29]:
# default rate by loan type
print(app.groupby("NAME_CONTRACT_TYPE")["TARGET"].mean())

NAME_CONTRACT_TYPE
Cash loans         0.083459
Revolving loans    0.054783
Name: TARGET, dtype: float64


### Correlation with TARGET

| Column | Correlation |
|---|---|
| EXT_SOURCE_3 | -0.179 |
| EXT_SOURCE_2 | -0.160 |
| EXT_SOURCE_1 | -0.155 |
| AGE | -0.078 |
| AMT_CREDIT | -0.030 |
| AMT_INCOME_TOTAL | -0.004 |

- The three `EXT_SOURCE` columns (external credit scores) have the strongest relation with default. The sign is negative, so **a higher score means less chance of default**.
- Age is also negative, which matches the age group table above.
- Income is basically **uncorrelated** with default on its own (-0.004). Surprising, but a ratio like credit/income might say more, so I'll try building that.
- All the values are small (nothing above 0.18), so no single feature will do the job. I'll need the model to combine them.

### Default rate by loan type

| Contract type | Default rate |
|---|---|
| Cash loans | 8.3% |
| Revolving loans | 5.5% |

Cash loans default more than revolving loans.